In [1]:
from pathlib import Path
import sys
import pandas as pd

# 1. SETUP PATH & MODUL REUSABLE
# Mengambil path direktori induk (root project) dari posisi notebook/script saat ini
PROJECT_ROOT = Path.cwd().parent

# Menambahkan root project ke sys.path agar modul 'etl' bisa di-import
sys.path.append(str(PROJECT_ROOT))

# Meng-import fungsi engine PostgreSQL yang sudah kita buat di etl/db_connection.py
from etl.db_connection import get_engine

# Menginisialisasi koneksi database
engine = get_engine()


# 2. SQL QUERY (DATA EXTRACTION & DENORMALIZATION)
# Menggabungkan data transaksi (sales_transactions), toko (stores), 
# produk (products), dan kategori (categories) menjadi satu dataset utuh.
# Sekaligus menghitung kalkulasi dasar 'revenue' (qty * unit_price).
query = """
SELECT
    st.store_id,
    st.store_location,
    s.transaction_id,
    s.transaction_date,
    s.transaction_time,
    s.transaction_qty,
    p.product_id,
    p.product_detail,
    c.category_name,
    s.unit_price,
    s.transaction_qty * s.unit_price AS revenue
FROM sales_transactions s
JOIN stores st
    ON s.store_id = st.store_id
JOIN products p
    ON s.product_id = p.product_id
JOIN categories c
    ON p.category_id = c.category_id;
"""


# 3. EKSEKUSI QUERY KE PANDAS DATAFRAME
# Membaca data dari PostgreSQL langsung menjadi Pandas DataFrame
sales_df = pd.read_sql(query, engine)


# 4. INSPEKSI AWAL DATASET (CHECK & VALIDATE)
# Menampilkan 5 baris pertama untuk melihat contoh isi tabel
sales_df.head()

# Ringkasan dimensi data dan total pendapatan untuk verifikasi awal
print(f"Total Baris Data  : {len(sales_df):,}")
print(f"Total Kolom Data  : {len(sales_df.columns)}")
print(f"Total Pendapatan  : Rp {sales_df['revenue'].sum():,.2f}")

Total Baris Data  : 149,116
Total Kolom Data  : 11
Total Pendapatan  : Rp 698,812.33
